<a href="https://colab.research.google.com/github/Yukselendincer/Microsoft-Data-Science/blob/main/dijital_i_kiz_bir_poc_rne_i.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Colab ortamına gerekli paketleri yüklüyoruz
!pip install -q streamlit plotly scikit-learn pandas numpy

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import time
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------
# 1. MAKİNE ÖĞRENMESİ MODELİ (Global Veri ile Eğitim)
# ---------------------------------------------------------
@st.cache_resource
def train_predictive_maintenance_model():
    """
    Python temelli global bir telemetri veri setini (Numenta NAB)
    kullanarak Kestirimci Bakım modeli eğitir.
    """
    # Global makine sıcaklığı arıza veri seti (Numenta Anomaly Benchmark)
    url = "https://raw.githubusercontent.com/numenta/NAB/master/data/realKnownCause/machine_temperature_system_failure.csv"
    df_global = pd.read_csv(url)

    n_samples = 2000
    # Global veriden gerçek sıcaklık sensör okumalarını çekiyoruz
    temp = df_global['value'].values[:n_samples]

    # Modele zenginlik katmak için diğer sensör parametrelerini sıcaklıkla korele ediyoruz
    np.random.seed(42)
    vib = (temp / 150) + np.random.normal(0, 0.05, n_samples)
    rpm = 1500 + (temp * 2) + np.random.normal(0, 50, n_samples)

    # Hedef Değişken (Arıza Var=1, Normal=0)
    failure = ((temp > 82) | (vib > 0.75) | ((temp > 78) & (vib > 0.65))).astype(int)

    df_train = pd.DataFrame({
        "Sicaklik_C": temp,
        "Titresim_mm_s": vib,
        "RPM": rpm,
        "Ariza": failure
    })

    X = df_train[["Sicaklik_C", "Titresim_mm_s", "RPM"]]
    y = df_train["Ariza"]

    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X, y)

    return model

# Modeli Önbelleğe Alıp Yüklüyoruz
model = train_predictive_maintenance_model()

# ---------------------------------------------------------
# 2. STREAMLIT ARAYÜZ VE SİMÜLASYON
# ---------------------------------------------------------
st.set_page_config(page_title="Colab Dijital İkiz ML PoC", page_icon="🤖", layout="wide")

st.title("🤖 Dijital İkiz & Scikit-Learn ML Kestirimci Bakım")
st.caption("Google Colab Üzerinde Canlı Telemetri ve Makine Öğrenmesi Tahmini")

# Yan Panel Kontrolleri
st.sidebar.header("🎛️ Simülasyon Ayarları")
refresh_rate = st.sidebar.slider("Yenileme Hızı (Saniye)", 0.5, 3.0, 1.0)
sample_size = st.sidebar.slider("Veri Geçmişi (Nokta)", 20, 100, 50)
inject_anomaly = st.sidebar.checkbox("⚠️ Yapay Arıza / Yük Bindir")

def generate_live_telemetry(n_points=50, anomaly=False):
    timestamps = pd.date_range(end=pd.Timestamp.now(), periods=n_points, freq="s")
    temp = 65.0 + np.random.normal(0, 1.5, n_points)
    vib = 0.4 + np.random.normal(0, 0.05, n_points)
    rpm = 1500 + np.random.normal(0, 15, n_points)

    if anomaly:
        temp[-8:] += np.linspace(15, 28, 8)
        vib[-8:] += np.linspace(0.3, 0.5, 8)
        rpm[-8:] -= np.linspace(100, 250, 8)

    return pd.DataFrame({
        "Zaman": timestamps,
        "Sicaklik_C": np.round(temp, 2),
        "Titresim_mm_s": np.round(vib, 3),
        "RPM": np.round(rpm, 1)
    })

# Streamlit'in yerleşik rerun mantığını kullanarak UI bloklanmasını önlüyoruz
placeholder = st.empty()
data = generate_live_telemetry(n_points=sample_size, anomaly=inject_anomaly)

# ---------------------------------------------------------
# 3. SCIKIT-LEARN İLE ANLIK TAHMİN (ML Inference)
# ---------------------------------------------------------
last_row = data.iloc[[-1]]
features = last_row[["Sicaklik_C", "Titresim_mm_s", "RPM"]]

# Model Tahmini
pred_class = model.predict(features)[0]
pred_prob = model.predict_proba(features)[0][1]

with placeholder.container():
    # Metrik Kartları
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Anlık Sıcaklık", f"{last_row['Sicaklik_C'].values[0]} °C")
    c2.metric("Anlık Titreşim", f"{last_row['Titresim_mm_s'].values[0]} mm/s")
    c3.metric("Motor Devri", f"{last_row['RPM'].values[0]} RPM")

    # ML Model Durum Göstergesi
    if pred_class == 1:
        c4.metric("ML Arıza Tahmini", "🚨 RİSKLİ", f"%{int(pred_prob*100)} Arıza Olasılığı")
    else:
        c4.metric("ML Arıza Tahmini", "✅ NORMAL", f"%{int((1-pred_prob)*100)} Sağlık Skoru")

    st.divider()

    # Grafikler
    g1, g2 = st.columns(2)
    with g1:
        st.subheader("🔥 Sıcaklık (°C)")
        fig_temp = px.line(data, x="Zaman", y="Sicaklik_C", markers=True)
        st.plotly_chart(fig_temp, use_container_width=True)

    with g2:
        st.subheader("📳 Titreşim (mm/s)")
        fig_vib = px.line(data, x="Zaman", y="Titresim_mm_s", markers=True)
        st.plotly_chart(fig_vib, use_container_width=True)

time.sleep(refresh_rate)
st.rerun()

In [ ]:
# Tünel IP adresinizi öğrenin (tunnel sayfasında şifre olarak runs.website IP'si istenir)
import urllib
print("Lütfen bu IP adresini kopyalayın:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip('\n'))

# Streamlit uygulamasını ve tüneli başlat
!streamlit run app.py &>/dev/null & npx localtunnel --port 8501

In [ ]:
!pkill -f streamlit
!pkill -f node